# 1-**Web Scraping**

To get the data to train on it our model

In [3]:
# Import libraries
import requests
from bs4 import BeautifulSoup
import csv
import re

In [2]:
# Get the data, Targeted [car model , KM , Prices]
url = "https://eg.hatla2ee.com/en/car/kia"

page = requests.get(url)
soup = BeautifulSoup(page.content, "html.parser")

cars = soup.find_all("div", class_="car")

def main(page):

  src = page.content
  response = requests.get(url)
  soup = BeautifulSoup(src, 'lxml')


  # List of dicts of our output data
  cars  = []

  # Get the data
  cars_list = soup.find_all("div", class_="newCarListUnit_data_wrap")

  ## get the KMs'
  meta_tags = soup.find_all('span', class_='newCarListUnit_metaTag')
  numbers = []

  if meta_tags:
      for tag in meta_tags:
          text = tag.get_text(strip=True)
          match = re.search(r'\d+(?:,\d+)?', text)
          number = float(match.group().replace(',', '')) if match else None
          numbers.append(number)
  else:
      numbers.append(None)
  for index in range(len(numbers)-1):
    if index % 2 != 0 :
      numbers[index] = 'highlight'
  while 'highlight' in numbers:
    numbers.remove('highlight')


  ## Get the car model
  a_tags = soup.find_all('div', class_='newCarListUnit_header')
  years = []

  for tag in a_tags:
      a_tag = tag.find('a')
      if a_tag:
          text = a_tag.text.strip()
          # استخراج السنة من نهاية النص
          match = re.search(r"(\d+)$", text)
          year = int(match.group(1)) if match else None
          years.append(year)

  ## Get the prices
  price_tags = soup.find_all('div', class_='main_price')
  prices = []

  if price_tags:
      for tag in price_tags:
          a_tag = tag.find('a')
          if a_tag:
              text = a_tag.get_text(strip=True)
              match = re.search(r'\d+(?:,\d+)*', text)
              price = float(match.group().replace(',', '')) if match else None
              prices.append(price)
  else:
      prices.append(None)

  # Forward the data to CSV file
  for i in range(len(years)):
      car = {
          'Car_model': years[i] if i < len(years) else None,
          'KM_Traveled': numbers[i] if i < len(numbers) else None,
          'Prices': prices[i] if i < len(prices) else None
      }
      cars.append(car)

  # Write to CSV
  keys = cars[0].keys()
  with open('cars.csv', 'w', newline='') as output_file:
      dict_writer = csv.DictWriter(output_file, keys)
      dict_writer.writeheader()
      dict_writer.writerows(cars)


main(page)


# **EDA Time**

In [3]:
# Import libraries
import pandas as pd



In [4]:
df=pd.read_csv(r'/content/cars.csv')
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/cars.csv'

In [125]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Car_model    30 non-null     int64  
 1   KM_Traveled  20 non-null     float64
 2   Prices       30 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 852.0 bytes
None


We Have 10 null values So we will tend to make data cleaning


# **Data cleaning**

In [97]:
# Removing nulls
df = df.dropna()

# **Ai model to predict the new instant price**

In [109]:
# Import libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Prepare data to model
X = df.drop('Prices', axis=1)
y = df['Prices']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Models
model = RandomForestRegressor(random_state=42)

# Train
model.fit(X_train, y_train)

# Test
predictions = model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, predictions)

print(mae)





497362.5


In [111]:

# New data instant
new_data = pd.DataFrame({
    'Car_model': [2014],
    'KM_Traveled': [43000]
})

# prediction
predicted_score = model.predict(new_data)[0]
predicted_score

np.float64(1177100.0)

In [ ]:
import joblib

# Assuming `model` is your trained model
joblib.dump(model, 'kia_model.pkl')